##Camada gold - onde vamos aplicar as regras de negocio

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_silver = spark.read.table("retail_sales.silver.sales_clean")
display(df_silver.limit(3))

###vendas por mes

In [0]:
vendas_por_mes = (
    df_silver
    .groupBy("ano","mes")
    .agg(
        round(sum(col("vendas_calculado")), 2).alias("vendas_total"),
        sum(col("quantidade_pedida")).alias("quantidade_total")
        
    )
)


###vendas por status

In [0]:
vendas_por_status = (
    df_silver
    .groupBy("status_pedido")
    .agg(
        countDistinct("numero_pedido").alias("total_pedidos"),
        round(sum(col("vendas_calculado")), 2).alias("vendas_total"),
        sum(col("quantidade_pedida")).alias("quantidade_total"),
    )
)

display(vendas_por_status)

###Maiores vendas por ano

In [0]:
top_vendas = (
    df_silver
    .groupBy("ano")
    .agg(
        countDistinct("numero_pedido").alias("top_pedidos"),
        round(sum(col("vendas_calculado")), 2).alias("vendas_total"),
        sum(col("quantidade_pedida")).alias("quantidade_total"),
        count("numero_linha_pedido").alias("linhas"),
    )
)

display(top_vendas)


###Salvar em delta as tabelas que respondem as regras de negocio

In [0]:
vendas_por_mes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_sales.gold.vendas_por_mes")
vendas_por_status.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("retail_sales.gold.vendas_por_status")
top_vendas.write.format("delta").mode("overwrite").option("overwriteschema", "true").saveAsTable("retail_sales.gold.top_vendas")